# 2 · Understand and inspect a spline PSF
Before starting, read [README.md](README.md). Run cells in order. Example files are from DECODE’s experiment, not the Nanoimager. Do not interpret a successful run as validated microscopy results. Nothing trains or writes data unless you explicitly enable it.
A PSF is the image of a point-like emitter. A *cubic spline calibration* models how that spot changes with depth using a fluorescent bead z-stack. Calibration produces the `.mat` file that DECODE’s simulator uses; this differs from fitting the network to a movie.

### Obtain a relevant calibration
Ask the lab for a bead z-stack or existing SMAP `3dcal.mat`, for each optical channel, matching the original objective, optical path, colour, acquisition orientation, and imaging conditions; ask for z step/units and camera settings. If unavailable, discuss whether new calibration measurements are representative of the original acquisition. The example bead files are from DECODE’s paper, not the Nanoimager. The documented workflow uses SMAP’s **Analyze / sr3D / calibrate3DsplinePSF** plugin, even for 2D data (bidirectional fit). Inspect its quality and confirm orientation and units with the person who collected the images. [DECODE tutorial](https://decode.readthedocs.io/en/release-0.10/tutorial.html).

### Step-by-step in SMAP
1. Install SMAP on the lab PC and open the relevant raw bead z-stack.
2. Run `calibrate3DsplinePSF`; supply the correct z-step and calibration settings from the acquisition. Follow SMAP’s calibration interface and verify fits and residuals; this notebook cannot infer settings or validate the calibration for you.
3. Save the generated `3dcal.mat`; record channel, camera, wavelength, date, units and orientation. Repeat separately for the other channel where needed.
4. Return to notebook 01 and set `param.InOut.calibration_file` to the new path **only after** updating camera and simulation parameters.


In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import tifffile

# Change this on the lab PC. It must contain experimental_data_workflow/.
ROOT = Path.cwd().resolve()
if not (ROOT / 'experimental_data_workflow').exists() and (ROOT.parent / 'experimental_data_workflow').exists():
    ROOT = ROOT.parent
EXAMPLE = ROOT / 'experimental_data_workflow'
WORK = ROOT / 'decode_work'
print('Project:', ROOT, '| Example exists:', EXAMPLE.exists())


### Inspect an example bead stack
This is a read-only visual check, not spline fitting. The example beads do not have the same side-by-side channel layout as your recordings.


In [ ]:
bead_files = sorted((EXAMPLE / 'beads').glob('*/*.tif'))
print('Example bead stacks:', len(bead_files))
if not bead_files: raise FileNotFoundError('Copy the full experimental_data_workflow example folder to ROOT.')
beads_path = bead_files[0]
with tifffile.TiffFile(beads_path) as t:
    n = len(t.pages); print('File:', beads_path, '| pages:', n, '| first page:', t.pages[0].shape)
    images = [t.pages[i].asarray() for i in (0, n//2, n-1)]
fig, axes = plt.subplots(1,3,figsize=(14,4))
for ax, image, ix in zip(axes, images, (0,n//2,n-1)):
    ax.imshow(image, cmap='gray'); ax.set_title(f'z-plane {ix} (auto contrast)')
plt.show()


In [ ]:
from scipy.io import whosmat
calibration = EXAMPLE / 'spline_calibration_3dcal.mat'
print('Example calibration contents:', whosmat(calibration))
print('This file is an existing fit; inspecting it does not calibrate your microscope.')
